# `splits` — Train/Test Scene Alignment

Finds and aligns the S1, S2, GT, and GT_0_8 GeoTIFFs for a given split (`Train`/`Test`), keyed off the S2 files. Converted from `src/data/splits.py`.

In [1]:
import os
from pathlib import Path
from typing import List, Dict

def get_scene_files(dataset_root: str, split: str) -> List[Dict[str, str]]:
    """
    Find and align S1, S2, GT, and GT_0_8 files for a given split (Train/Test).
    Returns a list of dicts: {'name': '...', 'S1': '...', 'S2': '...', 'GT': '...', 'GT_0_8': '...'}
    """
    root = Path(dataset_root)
    split_dir = root / "tifs" / split

    s1_dir = split_dir / "S1"
    s2_dir = split_dir / "S2"
    gt_dir = split_dir / "GT"
    gt08_dir = split_dir / "GT_0_8"

    # We use S2 files as reference
    s2_files = sorted(list(s2_dir.glob("*.tif")))

    aligned = []
    for s2_file in s2_files:
        name = s2_file.name

        s1_path = s1_dir / name
        gt_path = gt_dir / name
        gt08_path = gt08_dir / name

        if not s1_path.exists():
            raise FileNotFoundError(f"Missing S1 file: {s1_path}")
        if not gt_path.exists():
            raise FileNotFoundError(f"Missing GT file: {gt_path}")
        if not gt08_path.exists():
            raise FileNotFoundError(f"Missing GT_0_8 file: {gt08_path}")

        aligned.append({
            'name': s2_file.stem,
            'S1': str(s1_path),
            'S2': str(s2_file),
            'GT': str(gt_path),
            'GT_0_8': str(gt08_path)
        })

    return aligned

print("splits loaded: get_scene_files")


splits loaded: get_scene_files


## Demo — align Train and Test scenes

In [2]:
import yaml, os
REPO = r"C:\Users\prajw\OneDrive - Amrita Vishwa Vidyapeetham\Desktop\SEM5\Deep Learning\flood-segmentation-sentinel12"
with open(os.path.join(REPO, "configs", "data.yaml")) as f:
    _cfg = yaml.safe_load(f)

train_scenes = get_scene_files(_cfg["dataset_root"], "Train")
test_scenes = get_scene_files(_cfg["dataset_root"], "Test")

def _site_counts(scenes):
    counts = {}
    for s in scenes:
        site = s["name"].split("_")[0]
        counts[site] = counts.get(site, 0) + 1
    return counts

print(f"Train: {len(train_scenes)} scenes {_site_counts(train_scenes)}")
for s in train_scenes:
    print(f"  {s['name']}")
print(f"\nTest: {len(test_scenes)} scenes {_site_counts(test_scenes)}")
for s in test_scenes:
    print(f"  {s['name']}")


Train: 14 scenes {'BNPI': 7, 'KNPI': 7}
  BNPI_20210223
  BNPI_20210310
  BNPI_20230208
  BNPI_20230213
  BNPI_20230519
  BNPI_20240109
  BNPI_20240203
  KNPI_20210218
  KNPI_20210223
  KNPI_20220119
  KNPI_20220213
  KNPI_20220310
  KNPI_20220325
  KNPI_20220723

Test: 9 scenes {'BNPI': 4, 'KNPI': 5}
  BNPI_20220208
  BNPI_20220325
  BNPI_20231125
  BNPI_20240129
  KNPI_20230213
  KNPI_20230603
  KNPI_20240104
  KNPI_20240129
  KNPI_20240203
